# Notebook 10 — Final Analysis & Presentation Preparation

This final notebook helps you:
1. Generate a comprehensive comparison between PD-L1 and your independent target
2. Create publication-quality figures for your presentation slides
3. Export a complete report CSV
4. Reflect on what you learned

**Companion wiki page:** [9.6 Final Presentation Guide](https://rucha1796.github.io/internship-bindcraft-2026/m9_06_presentation/)

---
> Run after completing Notebooks 01–09.

In [ ]:
!pip install py3Dmol matplotlib -q
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import py3Dmol
import os, glob, json

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

# ============================================================
TARGET_NAME = "MyTarget"    # same as in NB09
# ============================================================

WORK_DIR = "/content/drive/MyDrive/bindcraft"
PDL1_DIR = f"{WORK_DIR}/PDL1_8aok"
TARGET_DIR = f"{WORK_DIR}/{TARGET_NAME}"

print("Configuration:")
print(f"  PD-L1 dataset:   {PDL1_DIR}")
print(f"  Your target:     {TARGET_DIR}")
print(f"  PD-L1 CSV:       {'✓ found' if os.path.exists(f'{PDL1_DIR}/final_design_stats.csv') else '✗ not found'}")
print(f"  Target CSV:      {'✓ found' if os.path.exists(f'{TARGET_DIR}/final_design_stats.csv') else '✗ not found'}")

## Cell 1 — Load both datasets

In [ ]:
def load_dataset(directory, label):
    csv = f"{directory}/final_design_stats.csv"
    if os.path.exists(csv):
        df = pd.read_csv(csv).sort_values("Average_i_pTM", ascending=False).reset_index(drop=True)
        df["dataset"] = label
        print(f"✓ {label}: {len(df)} accepted designs")
        return df
    else:
        # Generate demo data
        print(f"  {label} CSV not found — generating demo data")
        np.random.seed(42 if label == "PDL1" else 99)
        n = 12 if label == "PDL1" else 8
        scale = 1.0 if label == "PDL1" else 0.85  # Slightly different for variety
        df = pd.DataFrame({
            "binder_name": [f"{label}_t{i}_mpnn1" for i in range(1, n+1)],
            "binder_sequence": ["MKVIFGLMVGGVIAK" + "AEK" * i for i in range(1, n+1)],
            "Average_i_pTM": np.random.uniform(0.56*scale, 0.74, n),
            "Average_pLDDT": np.random.uniform(0.86, 0.95, n),
            "Average_Binder_pLDDT": np.random.uniform(0.81, 0.92, n),
            "Average_Binder_RMSD": np.random.uniform(0.5, 1.9, n),
            "Average_ShapeComplementarity": np.random.uniform(0.55*scale, 0.72, n),
            "Average_dG": np.random.uniform(-24, -10, n),
            "Average_i_pAE": np.random.uniform(5, 9.5, n),
            "Average_n_InterfaceHbonds": np.random.randint(3, 13, n).astype(float),
            "Average_Relaxed_Clashes": np.random.uniform(0, 4.5, n),
            "dataset": label
        })
        return df

df_pdl1   = load_dataset(PDL1_DIR, "PDL1")
df_target = load_dataset(TARGET_DIR, TARGET_NAME)

## Cell 2 — Side-by-side metric comparison

In [ ]:
METRICS = [
    ("Average_i_pTM", "i_pTM"),
    ("Average_ShapeComplementarity", "Shape Complementarity"),
    ("Average_Binder_RMSD", "Binder RMSD (Å)"),
    ("Average_dG", "dG (kcal/mol)"),
]
METRICS = [(c, l) for c, l in METRICS if c in df_pdl1.columns and c in df_target.columns]

fig, axes = plt.subplots(1, len(METRICS), figsize=(5*len(METRICS), 4))
if len(METRICS) == 1:
    axes = [axes]

THRESHOLDS = {
    "Average_i_pTM": 0.55,
    "Average_ShapeComplementarity": 0.55,
    "Average_Binder_RMSD": 2.0,
    "Average_dG": -10.0,
}

for ax, (col, label) in zip(axes, METRICS):
    pdl1_vals = df_pdl1[col].dropna()
    target_vals = df_target[col].dropna()
    
    all_vals = list(pdl1_vals) + list(target_vals)
    bins = np.linspace(min(all_vals), max(all_vals), 15)
    
    ax.hist(pdl1_vals, bins=bins, alpha=0.7, color="#3c5b6f",
            label=f"PDL1 (n={len(pdl1_vals)})", edgecolor="white")
    ax.hist(target_vals, bins=bins, alpha=0.7, color="#B76E79",
            label=f"{TARGET_NAME} (n={len(target_vals)})", edgecolor="white")
    
    if col in THRESHOLDS:
        ax.axvline(THRESHOLDS[col], color="black", linestyle="--",
                   linewidth=1.5, alpha=0.7, label="threshold")
    
    ax.set_title(label, fontsize=11)
    ax.legend(fontsize=8)

plt.suptitle(f"PD-L1 vs. {TARGET_NAME}: Metric Comparison",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("comparison_histograms.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: comparison_histograms.png — use this in your presentation")

## Cell 3 — Summary statistics table

In [ ]:
metrics_to_compare = [c for c, _ in METRICS]

rows = []
for col in metrics_to_compare:
    label = dict(METRICS)[col]
    pdl1_vals = df_pdl1[col].dropna()
    target_vals = df_target[col].dropna()
    rows.append({
        "Metric": label,
        f"PDL1 mean ± SD": f"{pdl1_vals.mean():.3f} ± {pdl1_vals.std():.3f}",
        f"{TARGET_NAME} mean ± SD": f"{target_vals.mean():.3f} ± {target_vals.std():.3f}",
        "Threshold": THRESHOLDS.get(col, "—")
    })

summary_df = pd.DataFrame(rows)
print("=" * 80)
print("METRIC COMPARISON SUMMARY")
print("=" * 80)
print(summary_df.to_string(index=False))
print()
print(f"  PDL1:      {len(df_pdl1)} accepted designs")
print(f"  {TARGET_NAME}: {len(df_target)} accepted designs")

## Cell 4 — Top design from each target

In [ ]:
for label, df, directory, chain_b_color in [
    ("PD-L1 #1", df_pdl1, PDL1_DIR, "#B76E79"),
    (f"{TARGET_NAME} #1", df_target, TARGET_DIR, "#8B5CF6")
]:
    # Find PDB
    pdbs = sorted(glob.glob(f"{directory}/Accepted/Ranked/*.pdb"))
    if pdbs:
        with open(pdbs[0]) as f:
            pdb_str = f.read()
        
        best = df.iloc[0]
        print(f"\n--- {label} ---")
        print(f"  i_pTM = {best.get('Average_i_pTM', 0):.3f} | ShapeComp = {best.get('Average_ShapeComplementarity', 0):.3f}")
        
        view = py3Dmol.view(width=700, height=450)
        view.addModel(pdb_str, "pdb")
        view.setStyle({"chain": "A"}, {"cartoon": {"color": "#3c5b6f"}})
        view.setStyle({"chain": "B"}, {"cartoon": {"color": chain_b_color}})
        view.addSurface(py3Dmol.VDW, {"opacity": 0.25, "color": "#3c5b6f"}, {"chain": "A"})
        view.zoomTo()
        view.setBackgroundColor("white")
        view.show()
    else:
        print(f"\n{label}: No PDB files found in {directory}/Accepted/Ranked/")

## Cell 5 — Presentation figure: the DMTA loop

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.axis('off')

# Draw DMTA cycle
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

steps = [
    (0.5, 0.85, "DESIGN",  "#3c5b6f", "BindCraft\nPD-L1 & " + TARGET_NAME + "\nWeeks 1–4 / 5–6"),
    (0.85, 0.5, "MAKE",    "#27ae60", "Gene synthesis\nE. coli expression\nProtein purification"),
    (0.5, 0.15, "TEST",    "#e67e22", "ELISA → SPR\nBinding kinetics\nKd measurement"),
    (0.15, 0.5, "ANALYZE", "#8B5CF6", "Which designs bind?\nWhich metrics predicted?\nImprove next round"),
]

for x, y, title, color, desc in steps:
    ax.add_patch(plt.Circle((x, y), 0.12, color=color, transform=ax.transAxes, zorder=3))
    ax.text(x, y + 0.005, title, ha="center", va="center", fontsize=12,
            fontweight="bold", color="white", transform=ax.transAxes, zorder=4)
    ax.text(x, y - 0.16 if y < 0.5 else y + 0.16, desc, ha="center",
            va="top" if y < 0.5 else "bottom", fontsize=8, color="#333333",
            transform=ax.transAxes, zorder=4)

# Arrows between steps
arrow_props = dict(arrowstyle="->", color="gray", lw=2)
for (x1, y1, *_), (x2, y2, *_) in zip(steps, steps[1:] + [steps[0]]):
    ax.annotate("", xytext=(x1, y1), xy=(x2, y2),
                arrowprops=arrow_props, xycoords="axes fraction",
                textcoords="axes fraction")

ax.set_title("The DMTA Cycle — Your Internship in Context",
             fontsize=14, fontweight="bold", pad=10)
ax.text(0.5, 0.5, "Your work\n(computational design)",
        ha="center", va="center", fontsize=9, color="#555555",
        style="italic", transform=ax.transAxes)

plt.tight_layout()
plt.savefig("dmta_cycle.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: dmta_cycle.png — use as a slide in your presentation")

## Cell 6 — Export complete report

In [ ]:
from datetime import date

# Combine datasets
combined = pd.concat([df_pdl1.head(5), df_target.head(5)], ignore_index=True)

report_cols = [c for c in [
    "dataset", "binder_name", "binder_sequence",
    "Average_i_pTM", "Average_ShapeComplementarity",
    "Average_Binder_RMSD", "Average_dG",
    "Average_n_InterfaceHbonds", "Average_Binder_pLDDT"
] if c in combined.columns]

report_path = f"internship_final_report_{date.today()}.csv"
combined[report_cols].to_csv(report_path, index=False)
print(f"✓ Saved: {report_path}")

# Print text version
print()
print("=" * 70)
print(f"INTERNSHIP FINAL REPORT — {date.today()}")
print("=" * 70)
print()
print(f"PD-L1 (teaching target):   {len(df_pdl1)} designs accepted")
print(f"{TARGET_NAME} (independent): {len(df_target)} designs accepted")
print()
print("Top 5 PD-L1 designs for synthesis:")
for i, row in df_pdl1.head(5).iterrows():
    print(f"  {i+1}. {row.get('binder_name','N/A')} | i_pTM={row.get('Average_i_pTM',0):.3f}")
print()
print(f"Top 5 {TARGET_NAME} designs for synthesis:")
for i, row in df_target.head(5).iterrows():
    print(f"  {i+1}. {row.get('binder_name','N/A')} | i_pTM={row.get('Average_i_pTM',0):.3f}")

try:
    from google.colab import files
    files.download(report_path)
except Exception:
    pass

## Cell 7 — Reflection

Edit this cell with your honest reflections. These will make your final presentation much stronger.

**What worked well:**
- [e.g., BindCraft found accepted designs consistently for PD-L1]
- [e.g., The i_pTM metric was easy to interpret]

**What was harder than expected:**
- [e.g., Understanding the PAE matrix at first]
- [e.g., Choosing hotspot residues for the independent target]

**What I would do differently:**
- [e.g., Run more trajectories for the independent target]
- [e.g., Try a wider length range]

**Questions the wet lab experiments will answer:**
- [e.g., Does i_pTM actually predict experimental binding?]
- [e.g., How many of the 5 selected designs will bind?]

**What I want to learn next:**
- [e.g., RFDiffusion for backbone generation]
- [e.g., Protein language models for sequence design]
- [e.g., Setting up a real protein purification experiment]

---
## You're done!

Files generated for your presentation:
- `comparison_histograms.png` — metric distributions comparing PD-L1 to your target
- `dmta_cycle.png` — the DMTA loop diagram
- `design_space.png` (from NB08) — scatter plot of your designs
- `internship_final_report_YYYY-MM-DD.csv` — full data table

3D screenshots for your slides:
- From **NB06**: screenshots of your top designs in Mol* (use Cmd+Shift+4 / Win+Shift+S)
- From **Cell 4** above: the top design from your independent target

**Presentation template:** See [Module 9.6 in the wiki](https://rucha1796.github.io/internship-bindcraft-2026/m9_06_presentation/) for the full 12-slide outline.

---
**Thank you for a great internship. Your designs go to the wet lab next!**